In [1]:
import pandas as pd

df = pd.read_csv("中間発表.csv")

display(df.head())

,日付,球場,球団,対戦相手,ホーム/ビジター,打者名,打順,守備位置,投手名,投手利き腕,...,打席前RE,打席後RE,RE24,打席前WE,打席後WE,WPA,WE推定方法_前,WE推定方法_後,WEサンプル数_前,WEサンプル数_後
0,2025/3/28,東京ドーム,巨人,ヤクルト,home,岡本 和真,4,3,奥川 恭伸,右,...,1.016,0.431,-0.585,0.714286,0.521739,-0.192547,exact,exact,14,23
1,2025/3/28,東京ドーム,ヤクルト,巨人,away,サンタナ,4,7,戸郷 翔征,右,...,0.365,0.189,-0.176,0.463250,0.432540,-0.030710,exact,exact,517,378
2,2025/3/28,東京ドーム,巨人,ヤクルト,home,岡本 和真,4,3,奥川 恭伸,右,...,0.189,0.074,-0.115,0.526906,0.505525,-0.021381,exact,exact,223,181
3,2025/3/28,東京ドーム,ヤクルト,巨人,away,サンタナ,4,7,戸郷 翔征,右,...,0.365,0.675,0.310,0.479167,0.632353,0.153186,exact,exact,192,34
4,2025/3/28,東京ドーム,ヤクルト,巨人,away,サンタナ,4,7,堀田 賢慎,右,...,0.365,0.365,1.000,0.871795,0.875000,0.003205,exact,exact,39,16


In [2]:
display(df[["日付", "球団", "打者名", "打席結果", "打点", "RE24", "WPA"]].head(10))

,日付,球団,打者名,打席結果,打点,RE24,WPA
0,2025/3/28,巨人,岡本 和真,三飛,0,-0.585,-0.192547
1,2025/3/28,ヤクルト,サンタナ,空三振,0,-0.176,-0.030710
2,2025/3/28,巨人,岡本 和真,空三振,0,-0.115,-0.021381
3,2025/3/28,ヤクルト,サンタナ,左安,0,0.310,0.153186
4,2025/3/28,ヤクルト,サンタナ,左本,1,1.000,0.003205
5,2025/3/28,巨人,岡本 和真,二ゴロ,0,-0.176,-0.001536
6,2025/3/28,ヤクルト,サンタナ,空三振,0,-0.176,-0.005911
7,2025/3/28,巨人,岡本 和真,一飛,0,-0.278,-0.087912
8,2025/3/28,巨人,岡本 和真,左安,0,0.613,0.190476
9,2025/3/28,中日,石川 昂弥,左飛,0,-0.184,-0.045517


In [4]:
import pandas as pd

# =========================
# 1. CSVを読み込む
# =========================
df = pd.read_csv("中間発表.csv")

# =========================
# 2. 数値列を整える
# =========================
numeric_cols = ["RE24", "WPA", "打点"]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# =========================
# 3. 本塁打列を作る
# =========================
if "打席結果" in df.columns:
    df["本塁打"] = df["打席結果"].astype(str).str.contains("本塁打|ホームラン|HR").astype(int)
else:
    df["本塁打"] = 0

# =========================
# 4. 選手別集計表 summary を作る
# =========================
summary = (
    df.groupby(["球団", "打者名"], as_index=False)
    .agg(
        打席数=("打者名", "count"),
        RE24合計=("RE24", "sum"),
        WPA合計=("WPA", "sum"),
        打点合計=("打点", "sum"),
        本塁打合計=("本塁打", "sum"),
    )
)

summary["1打席あたりRE24"] = summary["RE24合計"] / summary["打席数"]
summary["1打席あたりWPA"] = summary["WPA合計"] / summary["打席数"]

summary = summary.sort_values("RE24合計", ascending=False)

display(summary.head(10))

,球団,打者名,打席数,RE24合計,WPA合計,打点合計,本塁打合計,1打席あたりRE24,1打席あたりWPA
16,日本ハム,野村 佑希,20,6.100,0.914363,7,0,0.305000,0.045718
6,ヤクルト,オスナ,13,3.578,0.617753,2,0,0.275231,0.047519
9,ロッテ,ソト,17,3.355,-0.314895,5,0,0.197353,-0.018523
13,巨人,岡本 和真,31,1.916,0.278473,4,0,0.061806,0.008983
19,西武,中村 剛也,4,1.261,0.434562,0,0,0.315250,0.108641
20,阪神,森下 翔太,31,1.240,0.356859,4,0,0.040000,0.011512
4,オリックス,西野 真弘,1,1.012,-0.108702,1,0,1.012000,-0.108702
10,ロッテ,ポランコ,5,0.681,-0.041264,1,0,0.136200,-0.008253
7,ヤクルト,サンタナ,11,0.661,0.091890,1,0,0.060091,0.008354
11,中日,山本 泰寛,1,0.106,0.000000,0,0,0.106000,0.000000


In [5]:
display(
    summary.sort_values("RE24合計", ascending=False)
    .head(10)[
        ["球団", "打者名", "打席数", "RE24合計", "WPA合計", "打点合計", "本塁打合計", "1打席あたりRE24"]
    ]
)

,球団,打者名,打席数,RE24合計,WPA合計,打点合計,本塁打合計,1打席あたりRE24
16,日本ハム,野村 佑希,20,6.100,0.914363,7,0,0.305000
6,ヤクルト,オスナ,13,3.578,0.617753,2,0,0.275231
9,ロッテ,ソト,17,3.355,-0.314895,5,0,0.197353
13,巨人,岡本 和真,31,1.916,0.278473,4,0,0.061806
19,西武,中村 剛也,4,1.261,0.434562,0,0,0.315250
20,阪神,森下 翔太,31,1.240,0.356859,4,0,0.040000
4,オリックス,西野 真弘,1,1.012,-0.108702,1,0,1.012000
10,ロッテ,ポランコ,5,0.681,-0.041264,1,0,0.136200
7,ヤクルト,サンタナ,11,0.661,0.091890,1,0,0.060091
11,中日,山本 泰寛,1,0.106,0.000000,0,0,0.106000
